
# DOWNLOAD_HISTORICAL_VIA_DRIVE.ipynb

## ✅ Bulletproof fix for Earth Engine 10MB limit (CHIRPS + ERA5, 1981-2023)

This notebook uses **Earth Engine Export API** (`Export.table.toDrive`) only.

### Why this works
- `getInfo()` / client-side collection downloads can fail with **10MB response limit**.
- `Export.table.toDrive()` runs server-side and writes directly to Google Drive.
- This avoids the 10MB cap and supports very large tables.

### What you will do
1. Start CHIRPS yearly export tasks (1981-2023) to Google Drive.
2. Start ERA5-Land yearly export tasks (1981-2023) to Google Drive.
3. Monitor tasks until completed.
4. Download CSVs from Drive.
5. Upload CSVs to Colab and compute climatology.
6. Save final climatology files for `production_pipeline/data/historical/`.

> **Important:** Do NOT use `FeatureCollection.getInfo()` for historical bulk data.


In [ ]:

# Cell 1: Setup + Auth (run once per session)
import ee
import pandas as pd
from pathlib import Path
import glob

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='genai-bangladesh-drought-demo')

START_YEAR = 1981
END_YEAR = 2023
DRIVE_FOLDER = 'bangladesh_drought_historical'
EXPECTED_YEARS = END_YEAR - START_YEAR + 1

print('✅ Earth Engine initialized')
print(f'✅ Export range: {START_YEAR}-{END_YEAR} ({EXPECTED_YEARS} years)')
print(f'✅ Drive folder: {DRIVE_FOLDER}')


In [ ]:

# Cell 2: Load districts and build EE FeatureCollection
from production_pipeline.extractors.static_extractor import load_hdx_boundaries
from production_pipeline.extractors.historical_extractor import build_districts_ee_feature_collection

districts_gdf = load_hdx_boundaries()

required_cols = {'district_id_canonical', 'district_name_canonical', 'geometry'}
missing = sorted(required_cols - set(districts_gdf.columns))
if missing:
    raise ValueError(f'HDX boundaries missing required columns: {missing}')

if len(districts_gdf) < 64:
    print(f'⚠️ WARNING: Expected ~64 districts, found {len(districts_gdf)}')

# Robust conversion (handles polygon/multipolygon safely)
districts_fc = build_districts_ee_feature_collection(districts_gdf)

print(f'✅ Districts loaded: {len(districts_gdf)}')
print('✅ EE FeatureCollection created for exports')


In [ ]:

# Cell 3: Export CHIRPS one year -> Google Drive (NO 10MB LIMIT)
def export_chirps_year(year, districts_fc, folder=DRIVE_FOLDER):
    """Export one year of CHIRPS daily district means to Drive CSV."""
    chirps = (
        ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
        .filterDate(f'{year}-01-01', f'{year + 1}-01-01')
        .select(['precipitation'])
    )

    def reduce_image(img):
        date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
        reduced = img.reduceRegions(
            collection=districts_fc,
            reducer=ee.Reducer.mean(),
            scale=5566
        )

        def add_fields(feat):
            return feat.set({
                'date': date,
                'rainfall_mm': feat.get('mean'),
                'dataset': 'CHIRPS',
                'year': year
            })

        return reduced.map(add_fields)

    all_data = ee.FeatureCollection(chirps.map(reduce_image).flatten())

    task = ee.batch.Export.table.toDrive(
        collection=all_data,
        description=f'chirps_{year}',
        folder=folder,
        fileNamePrefix=f'chirps_{year}',
        fileFormat='CSV'
    )
    task.start()
    return task


In [ ]:

# Cell 4: Start CHIRPS exports for all years (1981-2023)
print('Starting CHIRPS exports to Google Drive...')
chirps_tasks = []

for year in range(START_YEAR, END_YEAR + 1):
    task = export_chirps_year(year, districts_fc)
    chirps_tasks.append((year, task))
    print(f'✅ Started export: CHIRPS {year}')

print('
' + '='*80)
print('📋 CHIRPS NEXT STEPS')
print('='*80)
print('1) Open: https://code.earthengine.google.com/tasks')
print('2) Confirm tasks chirps_1981 ... chirps_2023 are running')
print(f'3) Output folder in Drive: {DRIVE_FOLDER}')
print('4) Wait until tasks complete')


In [ ]:

# Cell 5: Export ERA5-Land one year -> Google Drive (NO 10MB LIMIT)
def export_era5_year(year, districts_fc, folder=DRIVE_FOLDER):
    """Export one year of ERA5-Land daily district means to Drive CSV."""
    era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').filterDate(
        f'{year}-01-01', f'{year + 1}-01-01'
    )

    def prep_image(img):
        temp_mean = img.select('temperature_2m').subtract(273.15).rename('temp_mean_c')
        temp_min = img.select('temperature_2m_min').subtract(273.15).rename('temp_min_c')
        temp_max = img.select('temperature_2m_max').subtract(273.15).rename('temp_max_c')
        precip = img.select('total_precipitation_sum').multiply(1000).rename('precip_era5_mm')
        evap = img.select('total_evaporation_sum').multiply(1000).rename('evap_era5_mm')
        out = ee.Image.cat([temp_mean, temp_min, temp_max, precip, evap])
        return out.copyProperties(img, ['system:time_start'])

    prepared = era5.map(prep_image)

    def reduce_image(img):
        date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
        reduced = img.reduceRegions(
            collection=districts_fc,
            reducer=ee.Reducer.mean(),
            scale=11132
        )

        def add_fields(feat):
            return feat.set({
                'date': date,
                'dataset': 'ERA5_LAND',
                'year': year
            })

        return reduced.map(add_fields)

    all_data = ee.FeatureCollection(prepared.map(reduce_image).flatten())

    task = ee.batch.Export.table.toDrive(
        collection=all_data,
        description=f'era5_{year}',
        folder=folder,
        fileNamePrefix=f'era5_{year}',
        fileFormat='CSV'
    )
    task.start()
    return task


In [ ]:

# Cell 6: Start ERA5 exports for all years (1981-2023)
print('Starting ERA5-Land exports to Google Drive...')
era5_tasks = []

for year in range(START_YEAR, END_YEAR + 1):
    task = export_era5_year(year, districts_fc)
    era5_tasks.append((year, task))
    print(f'✅ Started export: ERA5 {year}')

print('
' + '='*80)
print('📋 ERA5 NEXT STEPS')
print('='*80)
print('1) Open: https://code.earthengine.google.com/tasks')
print('2) Confirm tasks era5_1981 ... era5_2023 are running')
print(f'3) Output folder in Drive: {DRIVE_FOLDER}')
print('4) Wait until tasks complete')


In [ ]:

# Cell 7: Check task progress (run repeatedly)
def check_export_status(start_year=START_YEAR, end_year=END_YEAR):
    tasks = ee.batch.Task.list()

    chirps_desc = {f'chirps_{y}' for y in range(start_year, end_year + 1)}
    era5_desc = {f'era5_{y}' for y in range(start_year, end_year + 1)}

    chirps_states = {}
    era5_states = {}

    for t in tasks:
        status = t.status() or {}
        desc = status.get('description', '')
        state = status.get('state', 'UNKNOWN')
        if desc in chirps_desc:
            chirps_states[state] = chirps_states.get(state, 0) + 1
        if desc in era5_desc:
            era5_states[state] = era5_states.get(state, 0) + 1

    def _completed(d):
        return d.get('COMPLETED', 0)

    print('📊 CHIRPS status:', chirps_states)
    print(f"   Progress: {_completed(chirps_states)}/{EXPECTED_YEARS}")

    print('📊 ERA5 status:', era5_states)
    print(f"   Progress: {_completed(era5_states)}/{EXPECTED_YEARS}")

    all_done = (_completed(chirps_states) == EXPECTED_YEARS and
                _completed(era5_states) == EXPECTED_YEARS)

    if all_done:
        print('
✅ ALL EXPORTS COMPLETE')
        print(f'📥 Download CSVs from Google Drive folder: {DRIVE_FOLDER}')
    else:
        print('
⏳ Exports still running. Re-run this cell after some time.')

    return all_done

check_export_status()



## 📥 Manual step after exports complete

1. Go to Google Drive → folder **`bangladesh_drought_historical`**.
2. Download all `chirps_*.csv` files and all `era5_*.csv` files.
3. In Colab, upload files to:
   - `/content/chirps_downloads/`
   - `/content/era5_downloads/`

Then run the next cells.


In [ ]:

# Cell 8: Process downloaded CHIRPS CSV files -> climatology
from production_pipeline.extractors.historical_extractor import compute_chirps_climatology

chirps_files = sorted(glob.glob('/content/chirps_downloads/chirps_*.csv'))
print(f'Found CHIRPS files: {len(chirps_files)}')

if len(chirps_files) == 0:
    raise FileNotFoundError('No CHIRPS files found at /content/chirps_downloads/chirps_*.csv')

chirps_frames = []
for f in chirps_files:
    df = pd.read_csv(f)
    if 'rainfall_mm' not in df.columns and 'mean' in df.columns:
        df = df.rename(columns={'mean': 'rainfall_mm'})
    chirps_frames.append(df)
    print(f"   ✅ Loaded {Path(f).name}: {len(df):,} rows")

chirps_combined = pd.concat(chirps_frames, ignore_index=True)
print(f'✅ CHIRPS combined rows: {len(chirps_combined):,}')

required_cols = {'date', 'district_id', 'rainfall_mm'}
missing = sorted(required_cols - set(chirps_combined.columns))
if missing:
    raise ValueError(f'CHIRPS combined data missing required columns: {missing}')

chirps_clim = compute_chirps_climatology(chirps_combined)

chirps_out = Path('/content/chirps_climatology_1981_2023.csv')
chirps_clim.to_csv(chirps_out, index=False)

print(f'✅ CHIRPS climatology rows: {len(chirps_clim):,}')
print(f'✅ Saved: {chirps_out}')


In [ ]:

# Cell 9: Process downloaded ERA5 CSV files -> climatology
from production_pipeline.extractors.historical_extractor import compute_era5_climatology

era5_files = sorted(glob.glob('/content/era5_downloads/era5_*.csv'))
print(f'Found ERA5 files: {len(era5_files)}')

if len(era5_files) == 0:
    raise FileNotFoundError('No ERA5 files found at /content/era5_downloads/era5_*.csv')

era5_frames = []
for f in era5_files:
    df = pd.read_csv(f)
    era5_frames.append(df)
    print(f"   ✅ Loaded {Path(f).name}: {len(df):,} rows")

era5_combined = pd.concat(era5_frames, ignore_index=True)
print(f'✅ ERA5 combined rows: {len(era5_combined):,}')

required_cols = {'date', 'district_id'}
missing = sorted(required_cols - set(era5_combined.columns))
if missing:
    raise ValueError(f'ERA5 combined data missing required columns: {missing}')

era5_clim = compute_era5_climatology(era5_combined)

era5_out = Path('/content/era5_climatology_1981_2023.csv')
era5_clim.to_csv(era5_out, index=False)

print(f'✅ ERA5 climatology rows: {len(era5_clim):,}')
print(f'✅ Saved: {era5_out}')


In [ ]:

# Cell 10: Bundle outputs into production_pipeline-style folder
bundle_dir = Path('/content/production_pipeline/data/historical')
bundle_dir.mkdir(parents=True, exist_ok=True)

chirps_target = bundle_dir / 'chirps_climatology_1981_2023.csv'
era5_target = bundle_dir / 'era5_climatology_1981_2023.csv'

pd.read_csv('/content/chirps_climatology_1981_2023.csv').to_csv(chirps_target, index=False)
pd.read_csv('/content/era5_climatology_1981_2023.csv').to_csv(era5_target, index=False)

print('✅ Bundle ready:')
print(f'   - {chirps_target}')
print(f'   - {era5_target}')
print('
📥 Download these 2 files and place them in: production_pipeline/data/historical/')



## Troubleshooting (fast)

- **If tasks show FAILED**: click failed task in GEE Tasks UI, inspect error, restart only failed years.
- **If quota limit appears**: run exports in smaller batches (e.g., 1981-1995, then next block).
- **If a year file is missing**: rerun only that year’s export function.
- **Never switch back to getInfo() for bulk extraction**; it can hit response-size limits.
